# 01 - Inventario del esquema origen

Objetivo: documentar las tablas, columnas, tipos de datos, claves y volumen de filas del esquema operacional `public`.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
tablas = q('''
select table_name
from information_schema.tables
where table_schema = 'public'
  and table_type = 'BASE TABLE'
order by table_name;
''')
tablas

In [ ]:
columnas = q('''
select table_name, ordinal_position, column_name, data_type, is_nullable
from information_schema.columns
where table_schema = 'public'
order by table_name, ordinal_position;
''')
columnas.head(30)

In [ ]:
counts = []
for table_name in tablas["table_name"]:
    value = q(f'select count(*) as filas from "{table_name}"').iloc[0, 0]
    counts.append({"tabla": table_name, "filas": int(value)})
pd.DataFrame(counts).sort_values("filas", ascending=False)

In [ ]:
q('''
select
    tc.table_name,
    kcu.column_name,
    ccu.table_name as references_table,
    ccu.column_name as references_column
from information_schema.table_constraints tc
join information_schema.key_column_usage kcu
  on tc.constraint_name = kcu.constraint_name
 and tc.table_schema = kcu.table_schema
join information_schema.constraint_column_usage ccu
  on ccu.constraint_name = tc.constraint_name
 and ccu.table_schema = tc.table_schema
where tc.constraint_type = 'FOREIGN KEY'
  and tc.table_schema = 'public'
order by tc.table_name, kcu.column_name;
''')